In [0]:
import sys
import types

# Mock the decorator so the import doesn't crash
mock_pipelines = types.ModuleType("pipelines")
mock_pipelines.table = lambda **kwargs: (lambda fn: fn)  # no-op decorator
sys.modules["pyspark.pipelines"] = mock_pipelines

In [0]:
# Databricks notebook source
# =============================================================
# Unit tests — bronze main_raw  (plain notebook, no pytest)
# =============================================================

from pyspark.sql import Row
from pyspark.sql.functions import col
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, TimestampType,
)

from src.pipelines.automotive_pipeline.transformations.bronze.bronze_main_raw import add_lineage


# -------------------------------------------------------------
# Helper — build the test DataFrame once
# -------------------------------------------------------------
def _bronze_df():
    schema = StructType([
        StructField("id",                  IntegerType(), True),
        StructField("_rescued_data",       StringType(),  True),
        StructField("_corrupt_record",     StringType(),  True),
        StructField("_metadata", StructType([
            StructField("file_path",              StringType(),    True),
            StructField("file_name",              StringType(),    True),
            StructField("file_modification_time", TimestampType(), True),
        ]), True),
    ])

    data = [
        Row(1,    None,              None,         Row("/path/a.csv", "a.csv", None)),  # valid
        Row(2,    '{"age":"thirty"}', None,        Row("/path/a.csv", "a.csv", None)),  # rescued
        Row(None, None,              '2,"bad,row', Row("/path/b.csv", "b.csv", None)),  # corrupt
    ]

    return spark.createDataFrame(data, schema)


# -------------------------------------------------------------
# Tests
# -------------------------------------------------------------
def test_lineage_columns_exist():
    out = add_lineage(_bronze_df())
    cols = out.columns
    assert "source_file_path"        in cols
    assert "source_file_name"        in cols
    assert "source_file_modified_at" in cols
    print("PASSED  test_lineage_columns_exist")


def test_rescued_rows_preserved():
    out = add_lineage(_bronze_df())
    assert out.filter(col("_rescued_data").isNotNull()).count() == 1
    print("PASSED  test_rescued_rows_preserved")


def test_corrupt_rows_preserved():
    out = add_lineage(_bronze_df())
    assert out.filter(col("_corrupt_record").isNotNull()).count() == 1
    print("PASSED  test_corrupt_rows_preserved")


def test_row_count_preserved():
    out = add_lineage(_bronze_df())
    assert out.count() == 3
    print("PASSED  test_row_count_preserved")


def test_row_not_both_corrupt_and_rescued():
    out = add_lineage(_bronze_df())
    bad = out.filter(
        col("_corrupt_record").isNotNull() &
        col("_rescued_data").isNotNull()
    ).count()
    assert bad == 0
    print("PASSED  test_row_not_both_corrupt_and_rescued")


# -------------------------------------------------------------
# Run all
# -------------------------------------------------------------
tests = [
    test_lineage_columns_exist,
    test_rescued_rows_preserved,
    test_corrupt_rows_preserved,
    test_row_count_preserved,
    test_row_not_both_corrupt_and_rescued,
]

passed, failed = 0, 0

for t in tests:
    try:
        t()
        passed += 1
    except Exception as e:
        print(f"FAILED  {t.__name__} — {e}")
        failed += 1

print(f"\n{passed} passed, {failed} failed out of {len(tests)} tests")